# 💎 ORIEN Neural Ecosystem: Dataset Analysis & Preprocessing
**Version**: 1.0 (Audit-to-Preprocessing Synergy)

This notebook performs a deep analysis of the acquired neural datasets and primes them for the high-fidelity training phase.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import collections

ROOT = Path('..').absolute()
DATASET_ROOT = ROOT / 'dataset'
print(f"[*] Dataset Root: {DATASET_ROOT}")

MODALITIES = {
    "FACE": DATASET_ROOT / "face/faces",
    "GESTURE": DATASET_ROOT / "gesture/classes",
    "EMOTION": DATASET_ROOT / "face_emotion/train"
}

## 🧬 Modality Analysis: Class Distributions

In [ ]:
def analyze_modality(name, path):
    print(f"[*] Analyzing {name}...")
    if not path.exists():
        print(f"  ⚠️ Path not found: {path}")
        return None
    
    classes = [d.name for d in path.iterdir() if d.is_dir()]
    counts = {}
    for cls in classes:
        cls_path = path / cls
        counts[cls] = len(list(cls_path.glob('*.jpg'))) + len(list(cls_path.glob('*.png')))
    
    df = pd.DataFrame(list(counts.items()), columns=['Class', 'Count']).sort_values('Count', ascending=False)
    return df

face_df = analyze_modality("FACE", MODALITIES["FACE"])
gesture_df = analyze_modality("GESTURE", MODALITIES["GESTURE"])
emotion_df = analyze_modality("EMOTION", MODALITIES["EMOTION"])

print("\n[OK] Analysis Complete.")

## 📊 Visualization: Synergy Report

In [ ]:
plt.figure(figsize=(15, 12))

# 1. Face (Top 20 identities)
plt.subplot(3, 1, 1)
sns.barplot(data=face_df.head(20), x='Count', y='Class', palette='viridis')
plt.title('FACE: Top 20 Identities (LFW)')

# 2. Gesture
plt.subplot(3, 1, 2)
sns.barplot(data=gesture_df, x='Count', y='Class', palette='magma')
plt.title('GESTURE: All Classes (HAGRID)')

# 3. Emotion
plt.subplot(3, 1, 3)
sns.barplot(data=emotion_df, x='Count', y='Class', palette='rocket')
plt.title('EMOTION: All Classes (FER2013)')

plt.tight_layout()
plt.show()

## ⚙️ Preprocessing Simulation
Testing image normalization and resizing parameters for the MobileNetV2 architecture.

In [ ]:
def test_preprocessing(modality_name, path, target_size=(96, 96)):
    print(f"[*] Sample Preprocessing for {modality_name}...")
    sample_class = next(path.iterdir())
    sample_img_path = next(sample_class.glob('*.jpg'))
    
    img = Image.open(sample_img_path)
    orig_size = img.size
    
    # Simulation: Resize + Grayscale check
    img_resized = img.resize(target_size)
    img_array = np.array(img_resized) / 255.0
    
    plt.imshow(img_resized)
    plt.title(f"{modality_name} Sample: {orig_size} -> {target_size}")
    plt.axis('off')
    plt.show()
    
    print(f"  Mean Pixel Value: {np.mean(img_array):.4f}")
    print(f"  Ready for Preprocessing Bridge: [YES]")

test_preprocessing("GESTURE", MODALITIES["GESTURE"])
test_preprocessing("EMOTION", MODALITIES["EMOTION"])

## 🛠️ Data Integrity Fix: Purging Corrupt Files
Scanning for broken headers that might crash the training pipeline.

In [ ]:
print("[*] Scanning for corrupt neural shards...")
bad_files = 0
for name, path in MODALITIES.items():
    for img_path in path.rglob('*.jpg'):
        try:
            with Image.open(img_path) as im:
                im.verify()
        except:
            print(f"  ⚠️ Corrupt file removed: {img_path}")
            os.remove(img_path)
            bad_files += 1

print(f"\n[OK] Integrity scan complete. Purged {bad_files} shards.")
print("💎 SYSTEM READY FOR PREPROCESSING BRIDGE.")